# BPMN Dataset Pipeline

Builds a train/eval dataset of BPMN 2.0 processes from a raw CSV export, converting each into the process schema used for redesign modeling.

## Config

In [ ]:
NUM_PROCESSES_TO_SAMPLE = 3000

In [ ]:
import copy
import json
import logging
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import pandas as pd
from tqdm.auto import tqdm

try:
    from langdetect import detect, DetectorFactory
    DetectorFactory.seed = 0
    LANGDETECT_AVAILABLE = True
except ImportError:
    LANGDETECT_AVAILABLE = False

from connectivity_resolver import ConnectivityResolutionError
from dataset_conversion import convert_to_schema
from validation import validate_record

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)-8s | %(message)s", datefmt="%H:%M:%S")
logger = logging.getLogger("bpmn_pipeline")

In [ ]:
@dataclass
class Config:
    input_path: Path = Path(r"C:\Users\yousu\Downloads\SAP\sap_sam_2022\data")
    csv_glob: str = "*.csv"
    csv_sep: str = ","
    csv_encoding: str = "utf-8"
    chunksize: int = 300

    target_language: str = "English"
    required_stencilset_substring: str = "bpmn2.0"
    excluded_name_markers: tuple = ("choreography", "conversation")
    min_tasks: int = 2
    max_tasks: int = 200

    sample_size: int = NUM_PROCESSES_TO_SAMPLE
    random_seed: int = 42
    train_ratio: float = 0.8

    output_path: Path = Path(r"C:\Users\yousu\Downloads\SAP\sap_sam_2022\processed")
    train_dirname: str = "train"
    eval_dirname: str = "eval"
    metadata_filename: str = "metadata.json"
    stats_filename: str = "stats_report.json"

    synth_seed: int = 7
    default_currency: str = "USD"
    hourly_rate_range: tuple = (15, 120)
    process_time_range_min: tuple = (5, 240)
    rework_time_fraction_range: tuple = (0.05, 0.25)
    default_job_titles: tuple = ("Process Owner", "Department Manager", "Analyst",
                                  "Coordinator", "Specialist", "Clerk", "Supervisor")
    default_org_name: str = "Synthetic Org"
    default_process_category_id: int = 1
    default_process_category_name: str = "Uncategorized"


def make_config(**overrides) -> Config:
    cfg = Config(**overrides)
    (cfg.output_path / cfg.train_dirname).mkdir(parents=True, exist_ok=True)
    (cfg.output_path / cfg.eval_dirname).mkdir(parents=True, exist_ok=True)
    return cfg


CONFIG = make_config()
random.seed(CONFIG.random_seed)
logger.info("input_path=%s output_path=%s sample_size=%d", CONFIG.input_path, CONFIG.output_path, CONFIG.sample_size)

## Load, filter, convert, save

Streams the CSV, filters to valid English BPMN 2.0, converts each row into the process schema (real connectivity, not placeholder gateways), validates, and reservoir-samples to disk one record at a time. Checkpoints after every file for crash recovery.

In [ ]:
REQUIRED_COLUMNS = ["Revision ID", "Model ID", "Organization ID", "Datetime",
                    "Model JSON", "Description", "Name", "Type", "Namespace"]


def _is_valid_bpmn_json(raw_json, cfg):
    if not raw_json or not isinstance(raw_json, str):
        return None
    try:
        model = json.loads(raw_json)
    except (json.JSONDecodeError, TypeError):
        return None
    stencilset = model.get("stencilset", {}) or {}
    namespace = (stencilset.get("namespace") or "") + (stencilset.get("url") or "")
    if cfg.required_stencilset_substring not in namespace.lower():
        return None
    if model.get("stencil", {}).get("id") != "BPMNDiagram":
        return None
    return model


def _extract_language(model, name, description, cfg):
    declared = (model.get("properties", {}) or {}).get("language")
    text = f"{name or ''} {description or ''}".strip()
    if not LANGDETECT_AVAILABLE or len(text) < 3:
        return declared
    try:
        code_ = detect(text)
    except Exception:
        return declared
    detected = "English" if code_ == "en" else code_
    return detected if declared == cfg.target_language else detected


def _count_tasks(model):
    count = 0

    def walk(shapes):
        nonlocal count
        for shape in shapes or []:
            if (shape.get("stencil") or {}).get("id") == "Task":
                count += 1
            walk(shape.get("childShapes"))

    walk(model.get("childShapes"))
    return count

In [ ]:
import pickle
import gc
import shutil


def process_dataset(cfg: Config, resume: bool = True) -> dict:
    input_path = Path(cfg.input_path)
    csv_files = sorted(input_path.glob(cfg.csv_glob)) if input_path.is_dir() else [input_path]
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found at {cfg.input_path}")

    slots_dir = cfg.output_path / "_reservoir_slots"
    slots_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = cfg.output_path / "pipeline_checkpoint.pkl"

    if resume and checkpoint_path.exists():
        with open(checkpoint_path, "rb") as f:
            ckpt = pickle.load(f)
        stats, rng, processed_files, eligible_count = ckpt["stats"], ckpt["rng"], ckpt["processed_files"], ckpt["eligible_count"]
        logger.info("resuming: %d files done, %d eligible so far", len(processed_files), eligible_count)
    else:
        stats = {"scanned": 0, "bad_json": 0, "wrong_language": 0, "excluded_variant": 0,
                  "size_out_of_range": 0, "passed_filter": 0, "conversion_failures": 0,
                  "validation_failures": 0, "conversion_error_samples": [], "validation_error_samples": []}
        rng = random.Random(cfg.random_seed)
        processed_files = set()
        eligible_count = 0
        for f in slots_dir.glob("slot_*.json"):
            f.unlink()

    remaining = [p for p in csv_files if str(p) not in processed_files]

    for csv_path in remaining:
        try:
            reader = pd.read_csv(csv_path, sep=cfg.csv_sep, encoding=cfg.csv_encoding,
                                  usecols=lambda c: c in REQUIRED_COLUMNS,
                                  chunksize=cfg.chunksize, dtype=str, on_bad_lines="skip")
        except Exception as exc:
            logger.warning("skipping unreadable file %s: %s", csv_path.name, exc)
            processed_files.add(str(csv_path))
            continue

        for chunk in tqdm(reader, desc=f"Processing {csv_path.name}", unit="chunk"):
            chunk = chunk.fillna("")
            for _, row in chunk.iterrows():
                stats["scanned"] += 1
                name = row.get("Name") or ""
                description = row.get("Description") or ""

                if any(m in name.lower() for m in cfg.excluded_name_markers):
                    stats["excluded_variant"] = stats.get("excluded_variant", 0) + 1
                    continue

                model = _is_valid_bpmn_json(row.get("Model JSON"), cfg)
                if model is None:
                    stats["bad_json"] += 1
                    continue

                if _extract_language(model, name, description, cfg) != cfg.target_language:
                    stats["wrong_language"] += 1
                    continue

                n_tasks = _count_tasks(model)
                if not (cfg.min_tasks <= n_tasks <= cfg.max_tasks):
                    stats["size_out_of_range"] += 1
                    continue

                stats["passed_filter"] += 1
                raw_row = {"revision_id": row.get("Revision ID"), "model_id": row.get("Model ID"),
                           "organization_id": row.get("Organization ID"), "datetime": row.get("Datetime"),
                           "name": name, "description": description, "model": model}

                process_id = 100_000 + stats["scanned"]
                try:
                    record = convert_to_schema(raw_row, process_id, cfg)
                except Exception as exc:
                    stats["conversion_failures"] += 1
                    if len(stats["conversion_error_samples"]) < 10:
                        stats["conversion_error_samples"].append({"name": name, "error": str(exc)})
                    continue

                problems = validate_record(record)
                if problems:
                    stats["validation_failures"] += 1
                    if len(stats["validation_error_samples"]) < 10:
                        stats["validation_error_samples"].append({"process_id": process_id, "problems": problems})
                    continue

                eligible_count += 1
                if eligible_count <= cfg.sample_size:
                    slot_index = eligible_count - 1
                else:
                    j = rng.randint(0, eligible_count - 1)
                    if j >= cfg.sample_size:
                        continue
                    slot_index = j

                with open(slots_dir / f"slot_{slot_index:05d}.json", "w", encoding="utf-8") as f:
                    json.dump(record, f, indent=2, ensure_ascii=False)

        processed_files.add(str(csv_path))
        logger.info("finished %s | scanned=%d passed=%d eligible=%d reservoir=%d/%d",
                    csv_path.name, stats["scanned"], stats["passed_filter"], eligible_count,
                    min(eligible_count, cfg.sample_size), cfg.sample_size)

        with open(checkpoint_path, "wb") as f:
            pickle.dump({"stats": stats, "rng": rng, "processed_files": processed_files,
                         "eligible_count": eligible_count}, f)
        gc.collect()

    stats["eligible_total"] = eligible_count
    stats["reservoir_size"] = min(eligible_count, cfg.sample_size)
    if eligible_count < cfg.sample_size:
        logger.warning("only %d eligible records found (< sample_size=%d)", eligible_count, cfg.sample_size)
    return stats

## Split, save, report

In [ ]:
def finalize_split_and_report(cfg: Config, stats: dict) -> dict:
    slots_dir = cfg.output_path / "_reservoir_slots"
    slot_files = sorted(slots_dir.glob("slot_*.json"))

    rng = random.Random(cfg.random_seed)
    shuffled = slot_files[:]
    rng.shuffle(shuffled)
    split_idx = round(len(shuffled) * cfg.train_ratio)
    train_files, eval_files = shuffled[:split_idx], shuffled[split_idx:]

    manifest = {"train": [], "eval": []}
    n_tasks_list, n_gateways_list, proc_times = [], [], []

    for split_name, files in (("train", train_files), ("eval", eval_files)):
        out_dir = cfg.output_path / getattr(cfg, f"{split_name}_dirname")
        out_dir.mkdir(parents=True, exist_ok=True)
        for slot_path in tqdm(files, desc=f"Saving {split_name}", unit="file"):
            with open(slot_path, encoding="utf-8") as f:
                record = json.load(f)
            filename = f"{record['process_code']}.json"
            shutil.move(str(slot_path), str(out_dir / filename))
            manifest[split_name].append(filename)
            n_tasks_list.append(len(record["process_task"]))
            n_gateways_list.append(len(record["gateways"]))
            proc_times.extend(pt["task"]["expected_process_time"] for pt in record["process_task"])

    def avg(vals):
        return round(sum(vals) / len(vals), 2) if vals else 0

    metadata = {
        "config": {k: (str(v) if isinstance(v, Path) else v) for k, v in vars(cfg).items()},
        "counts": {"train": len(manifest["train"]), "eval": len(manifest["eval"]),
                   "total": len(manifest["train"]) + len(manifest["eval"])},
        "manifest": manifest, "generated_at": pd.Timestamp.now(tz="UTC").isoformat(),
    }
    with open(cfg.output_path / cfg.metadata_filename, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

    report = {
        "scan_and_conversion_stats": {k: v for k, v in stats.items() if not k.endswith("_samples")},
        "split_counts": metadata["counts"],
        "dataset_characteristics": {
            "avg_tasks_per_process": avg(n_tasks_list),
            "min_tasks_per_process": min(n_tasks_list, default=0),
            "max_tasks_per_process": max(n_tasks_list, default=0),
            "avg_gateways_per_process": avg(n_gateways_list),
            "avg_task_process_time_minutes": avg(proc_times),
        },
        "sample_conversion_errors": stats.get("conversion_error_samples", []),
        "sample_validation_errors": stats.get("validation_error_samples", []),
    }
    with open(cfg.output_path / cfg.stats_filename, "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2)

    shutil.rmtree(slots_dir, ignore_errors=True)
    (cfg.output_path / "pipeline_checkpoint.pkl").unlink(missing_ok=True)

    print(f"scanned={stats['scanned']} passed_filter={stats['passed_filter']} "
          f"conversion_failures={stats['conversion_failures']} validation_failures={stats['validation_failures']}")
    print(f"train={metadata['counts']['train']} eval={metadata['counts']['eval']}")
    print(f"avg_tasks={report['dataset_characteristics']['avg_tasks_per_process']} "
          f"avg_gateways={report['dataset_characteristics']['avg_gateways_per_process']}")
    print(f"report: {cfg.output_path / cfg.stats_filename}")
    return report


def run_pipeline(cfg: Config, resume: bool = True) -> dict:
    stats = process_dataset(cfg, resume=resume)
    if stats.get("reservoir_size", 0) == 0:
        logger.error("no eligible records produced")
        return {}
    return finalize_split_and_report(cfg, stats)

## Self-test

In [ ]:
import csv as _csv
import tempfile


def _flow(fid, target, label=""):
    return {"resourceId": fid, "properties": {"name": label}, "stencil": {"id": "SequenceFlow"},
            "outgoing": [], "target": {"resourceId": target}, "childShapes": []}


def _test_bpmn_model(lang="English", n_tasks=3):
    shapes = [{"resourceId": "start1", "properties": {"name": "Start"},
               "stencil": {"id": "StartNoneEvent"}, "outgoing": [{"resourceId": "f_start"}], "childShapes": []},
              _flow("f_start", "task1")]
    for i in range(1, n_tasks + 1):
        nxt = f"task{i+1}" if i < n_tasks else "gw1"
        fid = f"f_task{i}"
        shapes.append({"resourceId": f"task{i}", "properties": {"name": f"Task {i}"},
                       "stencil": {"id": "Task"}, "outgoing": [{"resourceId": fid}], "childShapes": []})
        shapes.append(_flow(fid, nxt))
    shapes += [
        {"resourceId": "gw1", "properties": {"name": "Decision?", "gatewaytype": "XOR"},
         "stencil": {"id": "Exclusive_Databased_Gateway"},
         "outgoing": [{"resourceId": "sf1"}, {"resourceId": "sf2"}], "childShapes": []},
        _flow("sf1", "end1", "Yes"),
        _flow("sf2", "end1", "No"),
        {"resourceId": "end1", "properties": {"name": "End"}, "stencil": {"id": "EndNoneEvent"},
         "outgoing": [], "childShapes": []},
    ]
    return {"resourceId": "canvas", "properties": {"language": lang}, "stencil": {"id": "BPMNDiagram"},
            "stencilset": {"namespace": "http://b3mn.org/stencilset/bpmn2.0#"}, "childShapes": shapes}


def _build_test_csv(path, n_rows=5, prefix=""):
    rows, kept = [], 0
    for i in range(n_rows):
        rows.append({"Revision ID": f"rev{prefix}{i}", "Model ID": f"mod{prefix}{i}", "Organization ID": f"org{i}",
                     "Datetime": "2020-01-01 10:00:00", "Model JSON": json.dumps(_test_bpmn_model("English", 3 + i % 2)),
                     "Description": "test process", "Name": f"Test Process {prefix}{i}", "Type": "",
                     "Namespace": "http://b3mn.org/stencilset/bpmn2.0#"})
        kept += 1
    if prefix == "":
        rows.append({"Revision ID": "reves", "Model ID": "modes", "Organization ID": "orges",
                     "Datetime": "2020-01-01 10:00:00", "Model JSON": json.dumps(_test_bpmn_model("English", 3)),
                     "Description": "Elaborar productos para el cliente final", "Name": "Elaborar productos",
                     "Type": "", "Namespace": "http://b3mn.org/stencilset/bpmn2.0#"})
        rows.append({"Revision ID": "revmap", "Model ID": "modmap", "Organization ID": "orgmap",
                     "Datetime": "2020-01-01 10:00:00",
                     "Model JSON": json.dumps({"resourceId": "canvas", "stencil": {"id": "Diagram"},
                                               "stencilset": {"namespace": "http://www.signavio.com/stencilsets/processmap#"},
                                               "childShapes": []}),
                     "Description": "", "Name": "Process Map", "Type": "",
                     "Namespace": "http://www.signavio.com/stencilsets/processmap#"})
        rows.append({"Revision ID": "revbad", "Model ID": "modbad", "Organization ID": "orgbad",
                     "Datetime": "2020-01-01 10:00:00", "Model JSON": "{not valid json",
                     "Description": "", "Name": "Bad Process", "Type": "", "Namespace": ""})
        rows.append({"Revision ID": "revchor", "Model ID": "modchor", "Organization ID": "orgchor",
                     "Datetime": "2020-01-01 10:00:00", "Model JSON": json.dumps(_test_bpmn_model("English", 2)),
                     "Description": "", "Name": "My Choreography Process", "Type": "",
                     "Namespace": "http://b3mn.org/stencilset/bpmn2.0#"})
        # dangling task -- must be rejected during conversion, not silently accepted
        broken = _test_bpmn_model("English", 2)
        for shape in broken["childShapes"]:
            if shape["resourceId"] == "task1":
                shape["outgoing"] = []
        rows.append({"Revision ID": "revdangle", "Model ID": "moddangle", "Organization ID": "orgd",
                     "Datetime": "2020-01-01 10:00:00", "Model JSON": json.dumps(broken),
                     "Description": "A process with a broken task connection for testing purposes",
                     "Name": "Dangling Process For Testing Purposes", "Type": "",
                     "Namespace": "http://b3mn.org/stencilset/bpmn2.0#"})

    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = _csv.DictWriter(f, fieldnames=REQUIRED_COLUMNS + ["Type"] if False else
                                  ["Revision ID", "Model ID", "Organization ID", "Datetime", "Model JSON",
                                   "Description", "Name", "Type", "Namespace"])
        writer.writeheader()
        writer.writerows(rows)
    return kept


def run_self_test() -> bool:
    tmp_dir = Path(tempfile.mkdtemp(prefix="bpmn_pipeline_selftest_"))
    checks = []
    try:
        expected_0 = _build_test_csv(tmp_dir / "0.csv", n_rows=5, prefix="")
        expected_1 = _build_test_csv(tmp_dir / "1.csv", n_rows=4, prefix="b")
        total_expected = expected_0 + expected_1

        test_cfg = make_config(input_path=tmp_dir, output_path=tmp_dir / "out", csv_sep=",",
                                sample_size=100, chunksize=3)

        original_convert = convert_to_schema
        crash_marker = {"triggered": False}

        def crashy_convert(raw_row, process_id, cfg):
            if raw_row["name"].startswith("Test Process b2") and not crash_marker["triggered"]:
                crash_marker["triggered"] = True
                raise KeyboardInterrupt("simulated crash")
            return original_convert(raw_row, process_id, cfg)

        globals()["convert_to_schema"] = crashy_convert
        crashed = False
        try:
            process_dataset(test_cfg, resume=True)
        except KeyboardInterrupt:
            crashed = True
        finally:
            globals()["convert_to_schema"] = original_convert

        checks.append(("crash triggered", crashed))
        checkpoint_path = test_cfg.output_path / "pipeline_checkpoint.pkl"
        checks.append(("checkpoint exists after crash", checkpoint_path.exists()))
        slots_dir = test_cfg.output_path / "_reservoir_slots"
        slots_before = list(slots_dir.glob("slot_*.json")) if slots_dir.exists() else []
        checks.append(("records saved before crash", len(slots_before) > 0))

        report = run_pipeline(test_cfg, resume=True)
        checks.append(("report non-empty after resume", bool(report)))

        stats = report.get("scan_and_conversion_stats", {})
        checks.append(("eligible_total matches expected", stats.get("eligible_total") == total_expected))
        checks.append(("wrong_language caught mislabeled row", stats.get("wrong_language", 0) >= 1))
        checks.append(("excluded_variant caught choreography", stats.get("excluded_variant", 0) == 1))
        checks.append(("bad_json caught malformed/non-BPMN rows", stats.get("bad_json", 0) == 2))
        checks.append(("dangling task rejected during conversion", stats.get("conversion_failures", 0) >= 1))
        checks.append(("zero validation failures on valid rows", stats.get("validation_failures", 0) == 0))
        checks.append(("split total matches eligible", report["split_counts"]["total"] == total_expected))
        checks.append(("checkpoint cleaned up", not checkpoint_path.exists()))
        checks.append(("slots dir cleaned up", not slots_dir.exists()))

        train_dir, eval_dir = test_cfg.output_path / "train", test_cfg.output_path / "eval"
        saved = list(train_dir.glob("*.json")) + list(eval_dir.glob("*.json"))
        checks.append(("output files on disk", len(saved) == total_expected))
        if saved:
            with open(saved[0], encoding="utf-8") as f:
                sample = json.load(f)
            checks.append(("saved record has required fields", all(k in sample for k in
                          ["process_id", "process_code", "process_name", "bpmn_xml", "gateways", "process_task"])))
            checks.append(("saved record passes validate_record", validate_record(sample) == []))

        checks.append(("metadata.json written", (test_cfg.output_path / "metadata.json").exists()))
        checks.append(("stats_report.json written", (test_cfg.output_path / "stats_report.json").exists()))

    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)

    passed = all(p for _, p in checks)
    for desc, p in checks:
        print(f"[{'PASS' if p else 'FAIL'}] {desc}")
    print("ALL PASSED" if passed else "FAILURES ABOVE")
    return passed


self_test_passed = run_self_test()

## Run

In [ ]:
assert self_test_passed
report = run_pipeline(CONFIG)